## Exercise: Saving and restoring models in Python

#### In this exercise, we save a trained model and restore it to make additional predictions on new inputs.

### Learning Objectives
#### By the end of this train, you should be able to:
##### - Use pickling to save and reuse previously trained models

### Exercises

#### We are provided with the `Crop_yield` dataset that contains various factors that could influence the yield of a particular crop across different regions.
#### We are interested in how some features, namely `Temperature`, `Rainfall`, `Fertilizer_Usage`, and `Pesticide_Usage`, influence the yield of the crop

In [49]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle

In [2]:
# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/Python/Crop_yield.csv")
df.head(5)

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


### Exercise 1

#### We begin by training and evaluating a multiple linear regression model to map the relationship between the features: `Temperature`, `Rainfall`, `Fertilizer_Usage`, and `Pesticide_Usage` and the response variable, `Yield`. This model will enable us to predict crop yields based on the given factors.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Region            1000 non-null   object 
 1   Temperature       1000 non-null   float64
 2   Rainfall          1000 non-null   float64
 3   Soil_Type         1000 non-null   object 
 4   Fertilizer_Usage  1000 non-null   float64
 5   Pesticide_Usage   1000 non-null   float64
 6   Irrigation        1000 non-null   int64  
 7   Crop_Variety      1000 non-null   object 
 8   Yield             1000 non-null   float64
dtypes: float64(5), int64(1), object(3)
memory usage: 70.4+ KB


In [31]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# numerical_cols = [col for col in numerical_cols if col != 'Yield']

In [32]:
df.head()

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


In [33]:
for col in categorical_cols:
    print(f'{col}: {df[col].unique()}')

Region: ['East' 'West' 'North' 'South']
Soil_Type: ['Clayey' 'Sandy' 'Loamy']
Crop_Variety: ['Variety B' 'Variety A' 'Variety C']


In [34]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [35]:
df_encoded

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety B,Crop_Variety_Variety C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,19.540123,543.669287,99.907202,44.913209,1,29.484676,1,0,0,0,0,0,1
996,26.090378,549.562256,237.636041,42.582608,1,28.190250,1,0,0,0,0,0,1
997,25.104737,756.628061,236.604958,19.356913,1,37.860206,0,0,1,0,0,0,0
998,19.604622,607.894675,86.379325,18.025284,0,32.638678,0,0,1,1,0,0,1


In [37]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Temperature             1000 non-null   float64
 1   Rainfall                1000 non-null   float64
 2   Fertilizer_Usage        1000 non-null   float64
 3   Pesticide_Usage         1000 non-null   float64
 4   Irrigation              1000 non-null   int64  
 5   Yield                   1000 non-null   float64
 6   Region_North            1000 non-null   int32  
 7   Region_South            1000 non-null   int32  
 8   Region_West             1000 non-null   int32  
 9   Soil_Type_Loamy         1000 non-null   int32  
 10  Soil_Type_Sandy         1000 non-null   int32  
 11  Crop_Variety_Variety B  1000 non-null   int32  
 12  Crop_Variety_Variety C  1000 non-null   int32  
dtypes: float64(5), int32(7), int64(1)
memory usage: 74.3 KB


In [24]:
numerical_cols

['Temperature',
 'Rainfall',
 'Fertilizer_Usage',
 'Pesticide_Usage',
 'Irrigation']

In [43]:
# X = df_encoded[[col for col in df_encoded.columns if col != 'Yield']]
X = df_encoded[['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage']]
y = df_encoded['Yield']

In [44]:
model = LinearRegression()

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [46]:
model.fit(X_train, y_train)

LinearRegression()

In [47]:
y_pred = model.predict(X_test)

In [52]:
# Evaluate model
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print('R² Score: ', r2)
print('MAE Val: ', mae)
print('MSE Val: ', mse)

R² Score:  0.9963871774411283
MAE Val:  0.47237203020108126
MSE Val:  0.3346060708725867


In [54]:
results = pd.DataFrame({
    'Actual yield': y_test,
    'Predicted yield': y_pred
})

print(results)

     Actual yield  Predicted yield
521     30.293004        29.806031
737     18.196937        18.788588
740     30.063595        29.404299
660     29.371347        28.884064
411     22.699114        23.385595
..            ...              ...
408     34.282618        35.142378
332     17.957911        18.298628
208     30.266366        29.306103
613     33.891452        34.508402
78      33.365607        33.035356

[200 rows x 2 columns]


### Exercise 2

#### We want to be able to conveniently retrieve and use the model for predictions in future. We therefore need to persist the trained model by saving it to a file.

#### Use the `pickle` library to serialise the model and store it in a file named `crop_yield_model.pkl`.

In [55]:
with open('crop_yield_model.pkl', 'wb') as file:
    pickle.dump(model, file)

### Exercise 3

#### We have received a new set of conditions for which we need to predict the crop yield. To accomplish this, we'll utilise the model we previously trained and saved.

#### New conditions:
##### - Temperature - `25`
##### - Rainfall - `150`
##### - Fertilizer Usage - `200`
##### - Pesticide Usage - `30`

#### Prepare the new data, load the saved model, and use it to make predictions on the given feature values

In [57]:
new_data = {
    'Temperature': [25],
    'Rainfall': [150],
    'Fertilizer_usage': [200],
    'Pesticide_usage': [30]
}

new_df = pd.DataFrame(new_data)

In [58]:
new_df

,Temperature,Rainfall,Fertilizer_usage,Pesticide_usage
0,25,150,200,30


In [64]:
with open('crop_yield_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)
    
new_predict = loaded_model.predict(new_df)

D:\Users\steph\anaconda3\envs\interactive_visuals_ALX\lib\site-packages\sklearn\base.py:493: FutureWarning: The feature names should match those that were passed during fit. Starting version 1.2, an error will be raised.
Feature names unseen at fit time:
- Fertilizer_usage
- Pesticide_usage
Feature names seen at fit time, yet now missing:
- Fertilizer_Usage
- Pesticide_Usage

  warnings.warn(message, FutureWarning)


In [65]:
new_predict

array([8.17286033])